In [0]:
ruta = "/Volumes/workspace/ef_precios_medicamentos/datos/sinteticos_1m.csv"
df = spark.read.csv(ruta, header=True, inferSchema=True)
print(spark.version)
print(df.count())

4.2.0
1000000


In [0]:
import time

def medir(nombre, accion, repeticiones=3):
    accion()  # calentamiento, no se cuenta
    tiempos = []
    for i in range(repeticiones):
        t0 = time.perf_counter()
        accion()
        tiempos.append(time.perf_counter() - t0)
    tiempos_ord = sorted(tiempos)
    print(nombre, "| tiempos (s):", [round(t, 2) for t in tiempos],
          "| mediana (s):", round(tiempos_ord[len(tiempos_ord) // 2], 2))
    return tiempos

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType, IntegerType)

esquema = StructType([
    StructField("codigo_producto", DoubleType(), True),
    StructField("nombre_comercial", StringType(), True),
    StructField("concentracion", StringType(), True),
    StructField("forma_farmaceutica", StringType(), True),
    StructField("presentacion", StringType(), True),
    StructField("registro_sanitario", StringType(), True),
    StructField("ifa", StringType(), True),
    StructField("titular", StringType(), True),
    StructField("fabricante", StringType(), True),
    StructField("pais", StringType(), True),
    StructField("clasificacion", StringType(), True),
    StructField("n_establecimientos", DoubleType(), True),
    StructField("precio_media", DoubleType(), True),
    StructField("precio_mediana", DoubleType(), True),
    StructField("precio_moda", DoubleType(), True),
    StructField("periodo", StringType(), True),
    StructField("mes", IntegerType(), True),
    StructField("nombre_mes", StringType(), True),
    StructField("trimestre", IntegerType(), True),
    StructField("anio", IntegerType(), True),
    StructField("variacion_mensual_pct", DoubleType(), True),
])

In [0]:
base = "/Volumes/workspace/ef_precios_medicamentos/datos/"
consulta = lambda d_: d_.groupBy("clasificacion", "pais").agg(F.avg("precio_media")).collect()

for etiqueta in ["100k", "500k", "1m"]:
    ruta_csv = base + "sinteticos_" + etiqueta + ".csv"
    medir("inferSchema_" + etiqueta, lambda: consulta(spark.read.csv(ruta_csv, header=True, inferSchema=True)))
    medir("explicito_" + etiqueta, lambda: consulta(spark.read.csv(ruta_csv, header=True, schema=esquema)))

inferSchema_100k | tiempos (s): [1.7, 1.52, 1.63] | mediana (s): 1.63
explicito_100k | tiempos (s): [0.78, 0.81, 0.83] | mediana (s): 0.81
inferSchema_500k | tiempos (s): [1.78, 2.17, 1.64] | mediana (s): 1.78
explicito_500k | tiempos (s): [0.94, 0.9, 0.96] | mediana (s): 0.94
inferSchema_1m | tiempos (s): [2.07, 1.93, 2.05] | mediana (s): 2.05
explicito_1m | tiempos (s): [1.06, 1.0, 0.96] | mediana (s): 1.0
